In [26]:
import os
from dotenv import load_dotenv

api_key = os.environ.get('OPENAI_API_KEY')

In [7]:
import fitz
import json
import pandas as pd
from pptx import Presentation
from PIL import Image
import pytesseract
from unstructured.partition.auto import partition

# ---------- PDF Parser ----------
def pdf_parser(file):
    doc = fitz.open(file)

    text = ""

    for page in doc:
        text = text + page.get_text()

    return {
        "text": text,
        "metadata": {
            "file_name": file.name
        }
    }

# ---------- CSV Parser ----------
def csv_parser(filepath):

    df = pd.read_csv(filepath)

    return {
        "text": df.to_string(index=False),
        "metadata": {
            "file_name": filepath.name
        }
    }

# ---------- Image Parser ----------
def image_parser(filepath):

    img = Image.open(filepath)
    text = pytesseract.image_to_string(image=img)

    return {
        "text": text,
        "metadata": {
            "file_name": filepath.name
        }
    }

# ---------- JSON Parser ----------
def json_parser(filepath):

    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    return {
        "text": json.dumps(data, indent=2),
        "metadata": {
            "file_name":filepath.name
        }
    }

# ---------- PPT Parser ----------
def ppt_parser(filepath):
    elements = partition(filename=str(filepath))

    text = "\n".join(
        element.text.strip()
        for element in elements
        if hasattr(element, "text") and element.text
    )

    return {
        "text": text,
        "metadata": {
            "file_name": filepath.name
        }
    }

In [8]:
from pathlib import Path
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

UPLOAD_DIR = Path('./data')

all_files = []

for i in UPLOAD_DIR.rglob("*"):
    if i.is_file():
        all_files.append(i)

print("Length of all files is:", len(all_files))

print(all_files)

PARSERS = {
    ".pdf": pdf_parser,
    ".json": json_parser,
    ".csv": csv_parser,
    ".ppt": ppt_parser,
    ".pptx": ppt_parser,
    ".png": image_parser,
    ".jpg": image_parser,
    ".jpeg": image_parser,
}

documents = []

for file in all_files:

    print(file)
    extension = file.suffix.lower()
    
    parser = PARSERS.get(extension)

    if parser is None:
        print(f"Skipping {file.name}")
        continue

    doc = parser(file)
    documents.append(doc)

documents

Length of all files is: 5
[WindowsPath('data/csv/contact_center_service_automation_synthetic.csv'), WindowsPath('data/image/ml.png'), WindowsPath('data/json/contact_center_service_automation_synthetic.json'), WindowsPath('data/pdf/NIPS-2017-attention-is-all-you-need-Paper.pdf'), WindowsPath('data/ppt/sample-1.pptx')]
data\csv\contact_center_service_automation_synthetic.csv
data\image\ml.png
data\json\contact_center_service_automation_synthetic.json
data\pdf\NIPS-2017-attention-is-all-you-need-Paper.pdf
data\ppt\sample-1.pptx


[{'text': 'ticket_id customer_id          created_at         resolved_at  channel language   product     issue_type priority agent_id  wait_time_sec  handle_time_sec  first_call_resolution  csat_score customer_sentiment      status  knowledge_article_used  llm_confidence  escalated region\nTKT000001   CUST28289 2026-04-27 09:41:00 2026-04-27 09:47:18    Voice    Hindi Broadband Password Reset      Low    AG103             30              348                      1         3.1           Positive   Escalated                       0            0.84          1  North\nTKT000002   CUST54597 2026-03-23 18:37:00 2026-03-23 18:48:40     Chat    Tamil  Internet  Network Issue Critical    AG104             11              689                      0         3.4            Neutral     Pending                       1            0.92          1  South\nTKT000003   CUST91070 2026-06-27 03:36:00 2026-06-27 03:52:20    Email    Hindi    Mobile  Network Issue      Low    AG122             85            

In [9]:
type(documents)

list

## **Chunking**

### **Text Splitter**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size= 600,
    chunk_overlap= 100,
    separators= ["\n\n", "\n", ". ", " ", ""]
)

### **Create Chunks**

In [24]:
from langchain_core.documents import Document

chunked_documents = []

for doc in documents:

    text = doc["text"]
    metadata = doc["metadata"]

    chunks = text_splitter.split_text(text)

    for idx, chunk in enumerate(chunks):
        chunk_metadata = {
            'chunk_id': idx,
            **metadata
        }
        chunked_documents.append(Document(page_content = chunk, metadata = chunk_metadata))

chunked_documents[0:2]


[Document(metadata={'chunk_id': 0, 'file_name': 'contact_center_service_automation_synthetic.csv'}, page_content='ticket_id customer_id          created_at         resolved_at  channel language   product     issue_type priority agent_id  wait_time_sec  handle_time_sec  first_call_resolution  csat_score customer_sentiment      status  knowledge_article_used  llm_confidence  escalated region\nTKT000001   CUST28289 2026-04-27 09:41:00 2026-04-27 09:47:18    Voice    Hindi Broadband Password Reset      Low    AG103             30              348                      1         3.1           Positive   Escalated                       0            0.84          1  North'),
 Document(metadata={'chunk_id': 1, 'file_name': 'contact_center_service_automation_synthetic.csv'}, page_content='TKT000002   CUST54597 2026-03-23 18:37:00 2026-03-23 18:48:40     Chat    Tamil  Internet  Network Issue Critical    AG104             11              689                      0         3.4            Neutral  

## **Vector DB**

In [33]:
from pymilvus import MilvusClient

# ---------- Connect to Milvus ----------

endpoint = os.environ.get("ZILLIZ_URI")
milvus_api_key = os.environ.get("ZILLIZ_TOKEN")
collection_name = "my_agentic_rag"

vector_db_client = MilvusClient(
    uri= endpoint,
    token= milvus_api_key
)

# ---------- Create Collection ----------

DIMENSION = 1536

if not vector_db_client.has_collection(collection_name):

    vector_db_client.create_collection(
        collection_name=collection_name,
        dimension=DIMENSION,
        metric_type="COSINE"
    )

print("Collection Ready")

Collection Ready


## **Embedding Model**

In [32]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model = "text-embedding-3-small"
)